# 17.3 轉回浮點會差多少？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**weight誤差很小，輸出誤差就必定同樣小嗎？

權重誤差會經過輸入累加到輸出；同樣權重誤差，在不同輸入上可能有不同影響。先比較一層，再擴大模型。

**把直覺寫成數學：**Δy=x·(W_hat−W)ᵀ；誤差取決於x與誤差方向。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import quantize_symmetric

w = torch.randn(3, 4)
q, scale = quantize_symmetric(w, 4)
restored = q * scale
x = torch.randn(2, 4)
print("權重MAE", (w - restored).abs().mean().item(), "輸出MAE", (x @ w.T - x @ restored.T).abs().mean().item())

**如何讀結果：**當前隨機輸入是數值demo，不代表所有真實activation。

**只改一件事：**只把x幅度乘10，觀察輸出誤差。
